# Week 13: The Forge

A framework built from tensors for creating neural networks.
It allows the user to simply design the forward pass and it can auto back-prop for you.
We have standard linear layers as well as some activation functions as options for using the framework.

### Part 1: The Autograd Tensor - Elijah

This is the bones of the framework: a tensor class which acts as a carrier for data, its creators,
id, and the operation that created it. It has functions built in to backprop when given gradients from
its children matrices. Only matrix addition is programmed for the tensor in this part.

In [2]:
import numpy as np


class Tensor:
    def __init__(self, data, autograd=False, creators=None,
                 creation_op=None, id=None):
        # Initialize values to what was given upon creation
        self.data = np.array(data)
        self.autograd = autograd
        self.creators = creators
        self.creation_op = creation_op

        # These fields are not given but are filled in later
        #  or are randomly autogenerated as self.id is
        self.grad = None
        if id is None:
            id = np.random.randint(0, 100000)
        self.id = id
        self.children = {}

        # If this has creators, increment their .children dicts by 1
        #   at this tensor's id key
        if self.creators is not None:
            for c in self.creators:
                c.children[self.id] = c.children.get(self.id, 0) + 1

    def all_children_accounted_for(self):
        # If all the children have a value of 0 return true
        return all(cnt == 0 for cnt in self.children.values())

    def __add__(self, other):
        # If both have autograd, record relevant information
        if self.autograd and other.autograd:
            return Tensor(self.data + other.data, autograd=True,
                          creators=[self, other], creation_op="add")
        # Otherwise just return a plain tensor
        else:
            return Tensor(self.data + other.data)

    # These two funcs do the same thing: return self.data as str not object address
    def __repr__(self):
        return str(self.data)

    def __str__(self):
        return str(self.data)

    def backward(self, grad=None, grad_origin=None):
        # if its a plain tensor, it shouldn't do anything here
        if not self.autograd:
            return

        # If this is the starting point, grad begins as ones
        if grad is None:
            grad = Tensor(np.ones_like(self.data))

        # If a child is reporting this
        if grad_origin is not None:
            # If the child's count is 0 it is double backpropping, which shouldn't happen
            if self.children[grad_origin.id] == 0:
                raise Exception("cannot backprop more than once")

            # Otherwise, decrement it's count
            else:
                self.children[grad_origin.id] -= 1

        # if self has no grad, then set it to grad because you can't add to none
        if self.grad is None:
            self.grad = grad
        else:
            # increment gradient
            self.grad += grad

        # The gradient should not have its own graph
        assert(grad.autograd == False)

        # If this has creators AND
        # If all the children are accounted for or there is no grad_origin(it was the first)
        #  then go backward into its creators
        if self.creators is not None and (self.all_children_accounted_for() or grad_origin is None):
            if self.creation_op == "add":
                self.creators[0].backward(Tensor(self.grad.data), self)
                self.creators[1].backward(Tensor(self.grad.data), self)


if __name__ == "__main__":
    a = Tensor([1, 2, 3, 4, 5], autograd=True)
    b = Tensor([2, 2, 2, 2, 2], autograd=True)
    c = Tensor([5, 4, 3, 2, 1], autograd=True)
    d = a + b
    e = b + c
    f = d + e
    f.backward(Tensor(np.array([1, 1, 1, 1, 1])))
    print(b.grad.data)  # [2 2 2 2 2]

    x = Tensor([1, 2, 3, 4, 5], autograd=True)
    y = Tensor([1, 1, 1, 1, 1], autograd=True)
    s = x + y  # not a leaf, and used twice below
    loss = (s + a) + (s + c)  # a, c: the tensors from the block above
    loss.backward(Tensor(np.array([1, 1, 1, 1, 1])))
    print(x.grad.data)  # [2 2 2 2 2]

[2 2 2 2 2]
[2 2 2 2 2]


### Part 2: The Operation Table - Kevin

This adds the rest of the rules for the other matrix operations of negation, subtraction, element multiplying,
matrix multiplying, transposing, summing over a dimension, and expanding along a dimension.

In [3]:
import numpy as np


class Tensor:
    def __init__(self, data, autograd=False, creators=None,
                 creation_op=None, id=None):
        # Initialize values to what was given upon creation
        self.data = np.array(data)
        self.autograd = autograd
        self.creators = creators
        self.creation_op = creation_op

        # These fields are not given but are filled in later
        #  or are randomly autogenerated as self.id is
        self.grad = None
        if id is None:
            id = np.random.randint(0, 100000)
        self.id = id
        self.children = {}

        # If this has creators, increment their .children dicts by 1
        #   at this tensor's id key
        if self.creators is not None:
            for c in self.creators:
                c.children[self.id] = c.children.get(self.id, 0) + 1

    def all_children_accounted_for(self):
        # If all the children have a value of 0 return true
        return all(cnt == 0 for cnt in self.children.values())

    def __add__(self, other):
        # If both have autograd, record relevant information
        if self.autograd and other.autograd:
            return Tensor(self.data + other.data, autograd=True,
                          creators=[self, other], creation_op="add")
        # Otherwise just return a plain tensor
        else:
            return Tensor(self.data + other.data)

    def __neg__(self):
        # If this tensor has autograd, record the negation op
        if self.autograd:
            return Tensor(self.data * -1, autograd=True,
                          creators=[self], creation_op="neg")
        # Otherwise just return a plain tensor
        else:
            return Tensor(self.data * -1)

    def __sub__(self, other):
        # If both have autograd, record relevant information
        if self.autograd and other.autograd:
            return Tensor(self.data - other.data, autograd=True,
                          creators=[self, other], creation_op="sub")
        # Otherwise just return a plain tensor
        else:
            return Tensor(self.data - other.data)

    def __mul__(self, other):
        # If both have autograd, record relevant information
        if self.autograd and other.autograd:
            return Tensor(self.data * other.data, autograd=True,
                          creators=[self, other], creation_op="mul")
        # Otherwise just return a plain tensor
        else:
            return Tensor(self.data * other.data)

    def mm(self, x):
        # If both have autograd, record the matrix multiplication
        if self.autograd and x.autograd:
            return Tensor(self.data.dot(x.data), autograd=True,
                          creators=[self, x], creation_op="mm")
        # Otherwise just return a plain tensor
        else:
            return Tensor(self.data.dot(x.data))

    def transpose(self):
        # If this tensor has autograd, record the transpose op
        if self.autograd:
            return Tensor(self.data.transpose(), autograd=True,
                          creators=[self], creation_op="transpose")
        # Otherwise just return a plain tensor
        else:
            return Tensor(self.data.transpose())

    def sum(self, dim):
        # Encode the summed dimension in the op name so backward knows which axis to expand
        if self.autograd:
            return Tensor(self.data.sum(dim), autograd=True,
                          creators=[self], creation_op="sum_" + str(dim))
        # Otherwise just return a plain tensor
        else:
            return Tensor(self.data.sum(dim))

    def expand(self, dim, copies):
        # Insert a new axis at dim and repeat the data copies times along it
        trans_cmd = list(range(0, len(self.data.shape)))
        trans_cmd.insert(dim, len(self.data.shape))
        new_data = self.data.repeat(copies).reshape(
            list(self.data.shape) + [copies]).transpose(trans_cmd)

        # Encode the expanded dimension in the op name so backward knows which axis to sum
        if self.autograd:
            return Tensor(new_data, autograd=True,
                          creators=[self], creation_op="expand_" + str(dim))
        # Otherwise just return a plain tensor
        else:
            return Tensor(new_data)


    # These two funcs do the same thing: return self.data as str not object address
    def __repr__(self):
        return str(self.data)

    def __str__(self):
        return str(self.data)

    def backward(self, grad=None, grad_origin=None):
        # if its a plain tensor, it shouldn't do anything here
        if not self.autograd:
            return

        # If this is the starting point, grad begins as ones
        if grad is None:
            grad = Tensor(np.ones_like(self.data))

        # If a child is reporting this
        if grad_origin is not None:
            # If the child's count is 0 it is double backpropping, which shouldn't happen
            if self.children[grad_origin.id] == 0:
                raise Exception("cannot backprop more than once")

            # Otherwise, decrement it's count
            else:
                self.children[grad_origin.id] -= 1

        # if self has no grad, then set it to grad because you can't add to none
        if self.grad is None:
            self.grad = grad
        else:
            # increment gradient
            self.grad += grad

        # The gradient should not have its own graph
        assert(grad.autograd == False)

        # If this has creators AND
        # If all the children are accounted for or there is no grad_origin(it was the first)
        #  then go backward into its creators
        if self.creators is not None and (self.all_children_accounted_for() or grad_origin is None):
            if self.creation_op == "add":
                self.creators[0].backward(Tensor(self.grad.data), self)
                self.creators[1].backward(Tensor(self.grad.data), self)

            elif self.creation_op == "neg":
                # Pass the negated gradient back to the creator
                self.creators[0].backward(self.grad.__neg__(), self)

            elif self.creation_op == "sub":
                # First creator gets positive grad, second creator gets negated grad
                self.creators[0].backward(Tensor(self.grad.data), self)
                self.creators[1].backward(self.grad.__neg__(), self)

            elif self.creation_op == "mul":
                # Each creator gets grad multiplied by the other creator's data
                c0, c1 = self.creators
                c0.backward(Tensor(self.grad.data * c1.data), self)
                c1.backward(Tensor(self.grad.data * c0.data), self)

            elif self.creation_op == "mm":
                # Compute matrix multiplication gradients on .data and wrap as plain Tensors
                c0, c1 = self.creators
                c0.backward(Tensor(self.grad.data.dot(c1.data.T)), self)
                c1.backward(Tensor(c0.data.T.dot(self.grad.data)), self)

            elif self.creation_op == "transpose":
                # Transpose the gradient back to match the creator's original shape
                self.creators[0].backward(self.grad.transpose(), self)

            elif self.creation_op == "tanh":
                # derivative of tanh is 1 - tanh(x)^2
                out = self.data
                grad = self.grad.data * (1 - out * out)
                self.creators[0].backward(Tensor(grad), self)

            elif self.creation_op == "sigmoid":
                # derivative of sigmoid is sigmoid(x) * (1 - sigmoid(x))
                out = self.data
                grad = self.grad.data * (out * (1 - out))
                self.creators[0].backward(Tensor(grad), self)

            elif "sum" in self.creation_op:
                # Read the summed axis back out and expand the gradient along that dim
                dim = int(self.creation_op.split("_")[1])
                ds = self.creators[0].data.shape[dim]
                self.creators[0].backward(self.grad.expand(dim, ds), self)

            elif "expand" in self.creation_op:
                # Read the expanded axis back out and sum the gradient over that dim
                dim = int(self.creation_op.split("_")[1])
                self.creators[0].backward(self.grad.sum(dim), self)


if __name__ == "__main__":
    a = Tensor([1, 2, 3, 4, 5], autograd=True)
    b = Tensor([2, 2, 2, 2, 2], autograd=True)
    c = Tensor([5, 4, 3, 2, 1], autograd=True)
    d = a + b
    e = b + c
    f = d + e
    f.backward(Tensor(np.array([1, 1, 1, 1, 1])))
    print(b.grad.data)  # [2 2 2 2 2]

    x = Tensor([1, 2, 3, 4, 5], autograd=True)
    y = Tensor([1, 1, 1, 1, 1], autograd=True)
    s = x + y  # not a leaf, and used twice below
    loss = (s + a) + (s + c)  # a, c: the tensors from the block above
    loss.backward(Tensor(np.array([1, 1, 1, 1, 1])))
    print(x.grad.data)  # [2 2 2 2 2]

    # Check non-leaf reuse through mm
    X = Tensor([[1., 2.], [3., 4.]], autograd=True)
    W = Tensor([[0.5, -0.5], [1., 0.25]], autograd=True)
    A = Tensor([[1., 0.], [2., 1.]], autograd=True)
    B = Tensor([[0., 1.], [1., 3.]], autograd=True)
    h = X.mm(W)
    (h.mm(A) + h.mm(B)).sum(0).sum(0).backward()
    print(W.grad.data.tolist())  # [[8.0, 28.0], [12.0, 42.0]]

[2 2 2 2 2]
[2 2 2 2 2]
[[8.0, 28.0], [12.0, 42.0]]


### Part 3 - Robert

Here we create the optimizer class which acts as the adjustment of all the parameters in the model as well as
a base layer class which holds parameters, a linear class which holds a weight and bias, and sequential class
which steps forward through a network of layers.


In [ ]:
from tensor import Tensor


class SGD:
    def __init__(self, parameters, alpha):
        # Stores a list of tensors
        self.parameters = parameters
        # Learning rate for the function
        self.alpha = alpha


    def step(self, zero=True):
        # Visits all parameters and will update the weights on the gradient. Basically new_weight = old_weight - gradient * learning_rate
        for p in self.parameters:
            p.data -= p.grad.data * self.alpha


            # Clears out gradients post update to prevent them from affecting the next iteration
            if zero:
                p.grad.data *= 0




# test function designed to make sure the above code works.
if __name__ == "__main__":


    from tensor import Tensor


    p = Tensor(10.0, autograd=True) # example tensor
    p.grad = Tensor(2.0) # assumed gradient of 2


    optimizer = SGD([p], alpha=0.1) # optimizer with an alpha of .1


    print("Before:", p.data) # expect 10


    optimizer.step()


    print("After:", p.data) # expect 9.8


Before: 10.0
After: 9.8


### Part 3 cont.

In [5]:
from tensor import Tensor
import numpy as np


# Base Class
class Layer:
    def __init__(self):
        self.parameters = list() # the Tensors SGD must visit


    def get_parameters(self):
        return self.parameters # returns parameters at the layer


# Dense layer
class Linear(Layer):
    def __init__(self, n_in, n_out):
        super().__init__()
        W = np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in) # Weight matrix to prevent activations from changing size too much mid training
        self.weight = Tensor(W, autograd=True) # weight matrix to be learned
        self.bias = Tensor(np.zeros(n_out), autograd=True) # Bias vector to be learned. Init as zeros
        self.parameters += [self.weight, self.bias]


    def forward(self, input):
        return input.mm(self.weight) + self.bias.expand(0, len(input.data)) # matrix multiplication


# Chain multiple layers together
class Sequential(Layer):
    def __init__(self, layers):
        super().__init__()
        self.layers = layers # ordered list of layers


    def forward(self, input):
        for layer in self.layers:
            input = layer.forward(input) # feed each into the next
        return input


    def get_parameters(self):
        params = [] # collect all params from every layer
        for l in self.layers:
            params += l.get_parameters() # gather every layer's weights
        return params




# Main function meant to test the above code
if __name__ == "__main__":


    # Build a small network:
    # 2 inputs > 3 hidden neurons > 1 output neuron
    model = Sequential([
        Linear(2, 3),
        Linear(3, 1)
    ])


    # Gather all trainable parameters
    params = model.get_parameters()


    print("Number of parameters:", len(params))
    print("Expected: 4")
    print()


    # Linear(2,3)
    #   weight -> (2,3)
    #   bias   -> (3,)
    #
    # Linear(3,1)
    #   weight -> (3,1)
    #   bias   -> (1,)
    #
    # Total trainable tensors = 4


    for i, p in enumerate(params):
        print(f"Parameter {i}:")
        print(p.data)
        print()


    # Create one sample containing two features
    x = Tensor(np.array([[1.0, 2.0]]), autograd=True)


    # Run the input through the network
    output = model.forward(x)


    print("Network output:")
    print(output.data)
    print()


    print("Output shape:")
    print(output.data.shape)
    print("Expected shape: (1, 1)")


Number of parameters: 4
Expected: 4

Parameter 0:
[[ 0.72589441 -2.17629611 -1.89463647]
 [ 0.77548757 -0.28265627  2.23585388]]

Parameter 1:
[0. 0. 0.]

Parameter 2:
[[-0.04865726]
 [-1.12173359]
 [-0.82998325]]

Parameter 3:
[0.]

Network output:
[[0.82564227]]

Output shape:
(1, 1)
Expected shape: (1, 1)


### Part 4 - Brady

Here we train a simple two layer linear model and prove that our code works.

In [7]:
import numpy as np
from tensor import Tensor
from layers import Linear, Sequential
from optim import SGD

np.random.seed(0)

data = Tensor([[0,0], [0,1], [1,0], [1,1]], autograd=True)
target = Tensor([[0], [1], [0], [1]], autograd=True)

model = Sequential([Linear(2, 3), Linear(3, 1)])
optim = SGD(parameters=model.get_parameters(), alpha=0.01)


# Train the model for 10 iterations

for i in range(10):
    pred = model.forward(data)

    loss = ((pred - target) * (pred - target)).sum(0)

    loss.backward()
    optim.step()

    print("Sequential iteration", i, "Loss:", loss.data)




np.random.seed(0)

# two weighted tensors
W1 = Tensor(np.random.randn(2, 3), autograd=True)
W2 = Tensor(np.random.randn(3, 1), autograd=True)



# Create an optimizer for the four trainable tensors
optim = SGD(parameters=[W1, W2], alpha=0.01)


# Train the MLP for 10 iterations

for i in range(10):

    # Forward pass: input to hidden layer
    hidden = data.mm(W1) 

    # Forward pass: hidden layer to output
    pred = hidden.mm(W2)

    # calculate the squared error
    loss = ((pred - target) * (pred - target)).sum(0)

    # backpropagation
    loss.backward()

    # update both weight tensors
    optim.step()

    # print the loss
    print("Bare MLP iteration", i, "Loss:", loss.data)


Sequential iteration 0 Loss: [2.33428272]
Sequential iteration 1 Loss: [1.50922784]
Sequential iteration 2 Loss: [0.98448854]
Sequential iteration 3 Loss: [0.6478277]
Sequential iteration 4 Loss: [0.43169175]
Sequential iteration 5 Loss: [0.29332533]
Sequential iteration 6 Loss: [0.20503331]
Sequential iteration 7 Loss: [0.14877319]
Sequential iteration 8 Loss: [0.11283963]
Sequential iteration 9 Loss: [0.08970536]
Bare MLP iteration 0 Loss: [31.56243513]
Bare MLP iteration 1 Loss: [13.82336914]
Bare MLP iteration 2 Loss: [9.28593901]
Bare MLP iteration 3 Loss: [6.5759718]
Bare MLP iteration 4 Loss: [4.78009147]
Bare MLP iteration 5 Loss: [3.53685475]
Bare MLP iteration 6 Loss: [2.64938307]
Bare MLP iteration 7 Loss: [2.00179902]
Bare MLP iteration 8 Loss: [1.52170376]
Bare MLP iteration 9 Loss: [1.16166886]


### Part 5 - Colson

We implement activation function versions of our layer class which allows our model to embrace nonlinearity
and our framework to be much stronger.


In [ ]:
class Tensor:

### Same code as above
###  this is what is added
    def tanh(self):
        # use tanh on every element
        out = np.tanh(self.data)
        if self.autograd:
            return Tensor(out, autograd=True,
                        creators=[self], creation_op="tanh")
        else:
            return Tensor(out)


    def sigmoid(self):
        # use sigmoid on every element
        out = 1 / (1 + np.exp(-self.data))
        if self.autograd:
            return Tensor(out, autograd=True,
                        creators=[self], creation_op="sigmoid")
        else:
            return Tensor(out)

In [ ]:
### Added to layers file

# tanh activation layer
class Tanh(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input.tanh()


# sigmoid activation layer
class Sigmoid(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input.sigmoid()


# mean squared error loss layer
class MSELoss(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, pred, target):
        return ((pred - target) * (pred - target)).sum(0)

In [9]:
from tensor import Tensor
from layers import Linear, Sequential, Tanh, Sigmoid, MSELoss
from optim import SGD
import numpy as np


np.random.seed(0)

# training inputs and expected outputs
data = Tensor([[0, 0], [0, 1], [1, 0], [1, 1]], autograd=True)
target = Tensor([[0], [1], [0], [1]], autograd=True)

# make the nonlinear neural network
model = Sequential([
    Linear(2, 3),
    Tanh(),
    Linear(3, 1),
    Sigmoid()
])

# loss function and optimizer
criterion = MSELoss()
optim = SGD(parameters=model.get_parameters(), alpha=1)

# train model for 10 iterations
losses = []

for i in range(10):
    # forward pass
    pred = model.forward(data)

    # calculate prediction error
    loss = criterion.forward(pred, target)

    # backpropagate gradients
    loss.backward()

    # update weights and clear gradients
    optim.step()

    # save and display loss
    losses.append(float(np.sum(loss.data)))
    print(f"Iteration {i}: Loss = {loss.data}")

Iteration 0: Loss = [1.06372865]
Iteration 1: Loss = [0.75148144]
Iteration 2: Loss = [0.57384259]
Iteration 3: Loss = [0.39574294]
Iteration 4: Loss = [0.2482279]
Iteration 5: Loss = [0.15515294]
Iteration 6: Loss = [0.10423398]
Iteration 7: Loss = [0.07571169]
Iteration 8: Loss = [0.05837623]
Iteration 9: Loss = [0.04700013]


### Part 6 - Kevin

Tests to prove that our functions work.

In [10]:
import numpy as np
from tensor import Tensor
from optim import SGD
from layers import Linear, Sequential, Tanh, Sigmoid, MSELoss


def test_multi_use_accumulation():
    # Part 1 acceptance test: a leaf tensor used twice must sum incoming gradients
    a = Tensor([1, 2, 3, 4, 5], autograd=True)
    b = Tensor([2, 2, 2, 2, 2], autograd=True)
    c = Tensor([5, 4, 3, 2, 1], autograd=True)
    d = a + b
    e = b + c
    f = d + e
    f.backward(Tensor(np.array([1, 1, 1, 1, 1])))
    assert np.array_equal(b.grad.data, np.array([2, 2, 2, 2, 2])), (
        f"Expected [2, 2, 2, 2, 2], got {b.grad.data}"
    )


def test_add_passes_gradient_through():
    # g = (a + b) + (c + d) should pass ones back to every leaf tensor
    a = Tensor([1.0, 2.0, 3.0], autograd=True)
    b = Tensor([4.0, 5.0, 6.0], autograd=True)
    c = Tensor([7.0, 8.0, 9.0], autograd=True)
    d = Tensor([10.0, 11.0, 12.0], autograd=True)
    g = (a + b) + (c + d)
    g.backward(Tensor(np.ones_like(g.data)))
    ones = np.ones_like(a.data)
    for leaf in (a, b, c, d):
        assert np.array_equal(leaf.grad.data, ones), (
            f"Expected {ones}, got {leaf.grad.data}"
        )


def test_numeric_gradient_check_mul_sum():
    # Numeric gradient check on y = (x * x).sum(0) testing mul and sum backward branches
    eps = 1e-5
    x_vals = np.array([2.0, 3.0])
    x = Tensor(x_vals.copy(), autograd=True)
    y = (x * x).sum(0)
    y.backward()

    analytic = x.grad.data.copy()
    assert np.allclose(analytic, np.array([4.0, 6.0]), atol=1e-6), (
        f"Expected [4.0, 6.0], got {analytic}"
    )

    numeric = np.zeros_like(x_vals)
    for i in range(len(x_vals)):
        x_plus = x_vals.copy()
        x_plus[i] += eps
        f_plus = float((Tensor(x_plus) * Tensor(x_plus)).sum(0).data)

        x_minus = x_vals.copy()
        x_minus[i] -= eps
        f_minus = float((Tensor(x_minus) * Tensor(x_minus)).sum(0).data)

        numeric[i] = (f_plus - f_minus) / (2 * eps)

    max_diff = float(np.max(np.abs(analytic - numeric)))
    print(f"    [Mul/Sum Grad Check] analytic={analytic}, numeric={numeric}, max_diff={max_diff:.2e}")
    assert max_diff < 1e-4, f"Max diff {max_diff} >= 1e-4"


def test_every_step_decreases_loss():
    # Part 4 Sequential model (seed=0, alpha=0.01): loss must strictly decrease over 10 steps
    np.random.seed(0)
    data = Tensor([[0, 0], [0, 1], [1, 0], [1, 1]], autograd=True)
    target = Tensor([[0], [1], [0], [1]], autograd=True)
    model = Sequential([Linear(2, 3), Linear(3, 1)])
    optim = SGD(parameters=model.get_parameters(), alpha=0.01)

    losses = []
    for _ in range(10):
        pred = model.forward(data)
        loss = ((pred - target) * (pred - target)).sum(0)
        loss.backward()
        optim.step()
        losses.append(float(loss.data.item()))

    for i in range(1, len(losses)):
        assert losses[i] < losses[i - 1], (
            f"Loss failed to decrease at step {i}: {losses[i]} >= {losses[i - 1]}"
        )


def test_shape_and_parameter_count():
    # Sequential([Linear(2,3), Tanh(), Linear(3,1), Sigmoid()]) must return 4 parameter Tensors
    model = Sequential([Linear(2, 3), Tanh(), Linear(3, 1), Sigmoid()])
    params = model.get_parameters()
    assert len(params) == 4, f"Expected 4 parameter tensors, got {len(params)}"

    expected_shapes = [(2, 3), (3,), (3, 1), (1,)]
    actual_shapes = [p.data.shape for p in params]
    assert actual_shapes == expected_shapes, (
        f"Expected shapes {expected_shapes}, got {actual_shapes}"
    )


def test_the_wait():
    # Non-leaf tensor s = x + y used twice: engine must wait until all children report
    a = Tensor([1, 2, 3, 4, 5], autograd=True)
    c = Tensor([5, 4, 3, 2, 1], autograd=True)
    x = Tensor([1, 2, 3, 4, 5], autograd=True)
    y = Tensor([1, 1, 1, 1, 1], autograd=True)
    s = x + y
    loss = (s + a) + (s + c)
    loss.backward(Tensor(np.array([1, 1, 1, 1, 1])))
    assert np.array_equal(x.grad.data, np.array([2, 2, 2, 2, 2])), (
        f"Expected [2, 2, 2, 2, 2], got {x.grad.data}"
    )


def test_reuse_through_mm():
    # Non-leaf tensor h = X.mm(W) used twice in matrix multiplications
    X = Tensor([[1., 2.], [3., 4.]], autograd=True)
    W = Tensor([[0.5, -0.5], [1., 0.25]], autograd=True)
    A = Tensor([[1., 0.], [2., 1.]], autograd=True)
    B = Tensor([[0., 1.], [1., 3.]], autograd=True)
    h = X.mm(W)
    (h.mm(A) + h.mm(B)).sum(0).sum(0).backward()
    expected = [[8.0, 28.0], [12.0, 42.0]]
    assert np.allclose(W.grad.data, expected), (
        f"Expected {expected}, got {W.grad.data.tolist()}"
    )


def test_whole_network_gradient_check():
    # Numeric gradient check across the entire Part 5 nonlinear network
    # Runs through mm, add, expand, tanh, sigmoid, sub, mul, and sum at once
    np.random.seed(0)
    data = Tensor([[0, 0], [0, 1], [1, 0], [1, 1]], autograd=True)
    target = Tensor([[0], [1], [0], [1]], autograd=True)
    model = Sequential([Linear(2, 3), Tanh(), Linear(3, 1), Sigmoid()])
    criterion = MSELoss()

    pred = model.forward(data)
    loss = criterion.forward(pred, target)
    loss.backward()

    # Nudge entry [0, 0] of the first Linear layer's weight matrix by +-eps
    eps = 1e-5
    w = model.layers[0].weight
    analytic_val = float(w.grad.data[0, 0])
    orig_val = float(w.data[0, 0])

    w.data[0, 0] = orig_val + eps
    loss_plus = float(criterion.forward(model.forward(data), target).data.item())

    w.data[0, 0] = orig_val - eps
    loss_minus = float(criterion.forward(model.forward(data), target).data.item())

    # Restore original weight
    w.data[0, 0] = orig_val

    numeric_val = (loss_plus - loss_minus) / (2 * eps)
    abs_diff = abs(analytic_val - numeric_val)
    print(f"    [Full Net Grad Check W1[0,0]] analytic={analytic_val:.8f}, numeric={numeric_val:.8f}, abs_diff={abs_diff:.2e}")
    assert abs_diff < 1e-4, f"Network gradient check failed: abs_diff {abs_diff} >= 1e-4"


if __name__ == "__main__":
    tests = [
        test_multi_use_accumulation,
        test_add_passes_gradient_through,
        test_numeric_gradient_check_mul_sum,
        test_every_step_decreases_loss,
        test_shape_and_parameter_count,
        test_the_wait,
        test_reuse_through_mm,
        test_whole_network_gradient_check,
    ]

    passed = 0
    for test_fn in tests:
        try:
            test_fn()
            print(f"PASS: {test_fn.__name__}")
            passed += 1
        except Exception as e:
            print(f"FAIL: {test_fn.__name__} -> {e}")

    print(f"\n{passed}/{len(tests)} tests passed.")

PASS: test_multi_use_accumulation
PASS: test_add_passes_gradient_through
    [Mul/Sum Grad Check] analytic=[4. 6.], numeric=[4. 6.], max_diff=3.93e-11
PASS: test_numeric_gradient_check_mul_sum
PASS: test_every_step_decreases_loss
PASS: test_shape_and_parameter_count
PASS: test_the_wait
PASS: test_reuse_through_mm
    [Full Net Grad Check W1[0,0]] analytic=-0.01228883, numeric=-0.01228883, abs_diff=9.61e-12
PASS: test_whole_network_gradient_check

8/8 tests passed.


### Part 7 - Elijah

An explanation of "removing the warning" and how that completely invalidates a model's ability without it being expressly obvious.

### Holocron -- Reading

This week
the fragment is shown twice: in gold as Vael Korrith wrote it, and in green as Vex left it, the
second stroke cut.
HOLOCRON NOTE — from the holocron of Vael Korrith — Fragment 9, as written
*“A technique is written in two strokes. The first is the power: what the act does in the world. The
second is the price: how the act is weighed, and corrected, and made answerable. To keep only the
first is not to gain speed. It is to build a thing that cannot be told it is wrong — and so will be wrong
forever, at full strength.”*

HOLOCRON NOTE — the same passage, as Vex left it (warnings removed)
*“A technique is written in two strokes. The first is the power: what the act does in the world.”*

*In the framework you just built, “removing the warning” has a precise meaning
— implementing an operation’s forward without its backward branch. Pick one operation from your
tensor.py (say mm) and describe, concretely, what would happen during training if its backward
branch were deleted: which gradients would go missing, which weights would stop learning, and
whether the forward pass would give any visible sign that something was wrong. Then connect it
to the story: Vex kept the power and removed the price. Name the engineering practice (from Part
6, or from your own PR review process) that would have caught his missing half before it shipped.*

#### Response

For the matrix mult operation, if the backward branch was removed from training, there would be no gradients or gradient updates that would happen during training! There would be no learning at all happening for any weights involved because there would be no implementation of calculating the gradients and passing it back through the matrices involved. And there would be no sign from the forward pass that the model was doing anything wrong because passing forward through the model *does not require learning,* so although the model can seemingly predict things, it's effectively completely worthless because it cannot *learn,* it just spits out garbage. The practice of verifying your model's abilities would catch this issue easily. You must verify that your model is actually changing its predictions over time (toward correctness) and not just repeating garbage answers. All it takes is a few runs to see if your model is *intelligent* and not what is effectively a for loop of repeated print statements.